# Treemap 
---

## Imports

In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json

In [120]:
path="../data/results_7.4.0-rc3.csv"
files=pd.read_csv(
    path, 
    converters={"subsystem":json.loads,
                "mailing_list": json.loads,
                "list_subsystem": json.loads
               }
)

pd.set_option("display.max_colwidth", None) #helps visualizing cell info
line='- - - - - - - - - - - - - - - - - - - - - '
display(files.head())

,directory,file,subsystem,mailing_list,maintainer,reviewer,list_subsystem
0,.,.clippy.toml,"[RUST, THE REST]","[rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org]","[{""email"":""ojeda@kernel.org"",""name"":""Miguel Ojeda"",""role"":""maintainer:RUST""}]","[{""email"":""boqun@kernel.org"",""name"":""Boqun Feng"",""role"":""reviewer:RUST""},{""email"":""gary@garyguo.net"",""name"":""Gary Guo"",""role"":""reviewer:RUST""},{""email"":""bjorn3_gh@protonmail.com"",""name"":""\""BjÃ¶rn Roy Baron\"""",""role"":""reviewer:RUST""},{""email"":""lossin@kernel.org"",""name"":""Benno Lossin"",""role"":""reviewer:RUST""},{""email"":""a.hindborg@kernel.org"",""name"":""Andreas Hindborg"",""role"":""reviewer:RUST""},{""email"":""aliceryhl@google.com"",""name"":""Alice Ryhl"",""role"":""reviewer:RUST""},{""email"":""tmgross@umich.edu"",""name"":""Trevor Gross"",""role"":""reviewer:RUST""},{""email"":""dakr@kernel.org"",""name"":""Danilo Krummrich"",""role"":""reviewer:RUST""},{""email"":""daniel.almeida@collabora.com"",""name"":""Daniel Almeida"",""role"":""reviewer:RUST""},{""email"":""tamird@kernel.org"",""name"":""Tamir Duberstein"",""role"":""reviewer:RUST""},{""email"":""acourbot@nvidia.com"",""name"":""Alexandre Courbot"",""role"":""reviewer:RUST""},{""email"":""work@onurozkan.dev"",""name"":""\""Onur Ãzkan\"""",""role"":""reviewer:RUST""}]","[{'email': 'rust-for-linux@vger.kernel.org', 'role': 'open list:RUST'}, {'email': 'linux-kernel@vger.kernel.org', 'role': 'open list'}]"
1,.,.get_maintainer.ignore,[THE REST],[linux-kernel@vger.kernel.org],[],[],"[{'email': 'linux-kernel@vger.kernel.org', 'role': 'open list'}]"
2,.,AGENTS.md,[THE REST],[linux-kernel@vger.kernel.org],[],[],"[{'email': 'linux-kernel@vger.kernel.org', 'role': 'open list'}]"
3,.,.rustfmt.toml,"[RUST, THE REST]","[rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org]","[{""email"":""ojeda@kernel.org"",""name"":""Miguel Ojeda"",""role"":""maintainer:RUST""}]","[{""email"":""boqun@kernel.org"",""name"":""Boqun Feng"",""role"":""reviewer:RUST""},{""email"":""gary@garyguo.net"",""name"":""Gary Guo"",""role"":""reviewer:RUST""},{""email"":""bjorn3_gh@protonmail.com"",""name"":""\""BjÃ¶rn Roy Baron\"""",""role"":""reviewer:RUST""},{""email"":""lossin@kernel.org"",""name"":""Benno Lossin"",""role"":""reviewer:RUST""},{""email"":""a.hindborg@kernel.org"",""name"":""Andreas Hindborg"",""role"":""reviewer:RUST""},{""email"":""aliceryhl@google.com"",""name"":""Alice Ryhl"",""role"":""reviewer:RUST""},{""email"":""tmgross@umich.edu"",""name"":""Trevor Gross"",""role"":""reviewer:RUST""},{""email"":""dakr@kernel.org"",""name"":""Danilo Krummrich"",""role"":""reviewer:RUST""},{""email"":""daniel.almeida@collabora.com"",""name"":""Daniel Almeida"",""role"":""reviewer:RUST""},{""email"":""tamird@kernel.org"",""name"":""Tamir Duberstein"",""role"":""reviewer:RUST""},{""email"":""acourbot@nvidia.com"",""name"":""Alexandre Courbot"",""role"":""reviewer:RUST""},{""email"":""work@onurozkan.dev"",""name"":""\""Onur Ãzkan\"""",""role"":""reviewer:RUST""}]","[{'email': 'rust-for-linux@vger.kernel.org', 'role': 'open list:RUST'}, {'email': 'linux-kernel@vger.kernel.org', 'role': 'open list'}]"
4,.,.gitattributes,[THE REST],[linux-kernel@vger.kernel.org],[],[],"[{'email': 'linux-kernel@vger.kernel.org', 'role': 'open list'}]"


## Document reproducibility

In [121]:
import sys, platform, re
from importlib.metadata import version

match=re.search(r'^[^_]*_(.*?)\.csv$', path)
kernel_v=match.group(1)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print(f"Kernel Version:", kernel_v)


for lib in ["pandas", "numpy", "matplotlib", "seaborn", "plotly"]:
    print(lib, ":", version(lib))


Python: 3.14.6
Platform: Linux-6.12.116+deb13-amd64-x86_64-with-glibc2.41
Kernel Version: 7.4.0-rc3
pandas : 3.0.5
numpy : 2.5.2
matplotlib : 3.11.1
seaborn : 0.13.2
plotly : 7.1.0


## Dataset treatment

In [122]:
original_df = files.copy()

#create a full path column
files['path']=files['directory']+'/'+files['file']
files['path']=files['path'].str.removeprefix('./')

#gets the number of subsystems
files['n_subsystem']=files['subsystem'].map(len , na_action='ignore')

files=(files
    .explode('subsystem')
    .dropna(subset=['list_subsystem'])
    .reset_index(drop=True)
)

In [123]:
#create a new dataset with the amiling list associated with each subsystem

list_subs_df=(
    original_df[['list_subsystem']]
    .explode('list_subsystem')
    .dropna(subset=['list_subsystem'])
    .reset_index(drop=True)
)

#get the subsystem name
list_subs_df['subsystem']=(
    list_subs_df['list_subsystem']
    .str.get('role')
    .str.replace(r'^[^:]+:?', '', regex=True)
    .replace('', 'THE REST')
)

#get the mailing list
list_subs_df['mailing_list']=list_subs_df['list_subsystem'].str.get('email')

list_subs_df=list_subs_df.drop(columns=['list_subsystem'])
list_subs_df=list_subs_df.drop_duplicates(ignore_index=True)

#to improve execution speed
list_subs_df['subsystem'] = list_subs_df['subsystem'].astype('category')
list_subs_df['mailing_list'] = list_subs_df['mailing_list'].astype('category')

display(list_subs_df)
list_subs_df.nunique()

,subsystem,mailing_list
0,RUST,rust-for-linux@vger.kernel.org
1,THE REST,linux-kernel@vger.kernel.org
2,LICENSES and SPDX stuff,linux-spdx@vger.kernel.org
3,DOCUMENTATION,linux-doc@vger.kernel.org
4,DRM ACCEL DRIVERS FOR INTEL VPU,dri-devel@lists.freedesktop.org
...,...,...
2880,BPF [SELFTESTS] (Test Runners & Infrastructure),bpf@vger.kernel.org
2881,CACHESTAT: PAGE CACHE STATS FOR A FILE,linux-mm@kvack.org
2882,RCUTORTURE TEST FRAMEWORK,rcu@vger.kernel.org
2883,KVM GUEST_MEMFD,kvm@vger.kernel.org


subsystem       2535
mailing_list     239
dtype: int64

In [124]:
files=files.drop(columns=['maintainer', 'reviewer', 'list_subsystem'])

In [125]:
files['mailing_list']=files['mailing_list'].apply(lambda x: ', '.join(x))
(files.head(10))

,directory,file,subsystem,mailing_list,path,n_subsystem
0,.,.clippy.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.clippy.toml,2
1,.,.clippy.toml,THE REST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.clippy.toml,2
2,.,.get_maintainer.ignore,THE REST,linux-kernel@vger.kernel.org,.get_maintainer.ignore,1
3,.,AGENTS.md,THE REST,linux-kernel@vger.kernel.org,AGENTS.md,1
4,.,.rustfmt.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.rustfmt.toml,2
5,.,.rustfmt.toml,THE REST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.rustfmt.toml,2
6,.,.gitattributes,THE REST,linux-kernel@vger.kernel.org,.gitattributes,1
7,.,.cocciconfig,THE REST,linux-kernel@vger.kernel.org,.cocciconfig,1
8,.,.clang-format,CLANG-FORMAT FILE,linux-kernel@vger.kernel.org,.clang-format,2
9,.,.clang-format,THE REST,linux-kernel@vger.kernel.org,.clang-format,2


In [126]:
print("N of files:")
print(original_df.shape[0])

print("N of files in THE REST:")
print(files[files['subsystem']=='THE REST'].shape[0])

#remove 'THE REST' from files already in another subsystem
n_in_the_rest=~((files['subsystem']== 'THE REST') & (files['n_subsystem'] >1))
files=files[n_in_the_rest].copy()
files = files.reset_index(drop=True)

# Recalculate actual subsystem count per file after removing THE REST
files['n_subsystem'] = files.groupby("path")['subsystem'].transform('count')

N of files:
97371
N of files in THE REST:
97371


In [127]:
display(files)
print("Número de linhas: ", files.shape[0])

,directory,file,subsystem,mailing_list,path,n_subsystem
0,.,.clippy.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.clippy.toml,1
1,.,.get_maintainer.ignore,THE REST,linux-kernel@vger.kernel.org,.get_maintainer.ignore,1
2,.,AGENTS.md,THE REST,linux-kernel@vger.kernel.org,AGENTS.md,1
3,.,.rustfmt.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.rustfmt.toml,1
4,.,.gitattributes,THE REST,linux-kernel@vger.kernel.org,.gitattributes,1
...,...,...,...,...,...,...
184275,virt/kvm,vfio.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/vfio.c,1
184276,virt/kvm,kvm_main.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/kvm_main.c,1
184277,virt/lib,Kconfig,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/Kconfig,1
184278,virt/lib,irqbypass.c,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/irqbypass.c,1


Número de linhas:  184280


## Treemap building

### first visualization

In [128]:
files['top_dir']=files['directory'].str.split('/').str[0]

#agregate files in the same subsystem and directory to speedup treemap processing
files_agg=(
    files.groupby(["subsystem", "top_dir", "directory"], as_index=False)
    .agg(file_count=("file","count"))
)

subs_lists=(
    list_subs_df.groupby("subsystem", as_index=False, observed=True)['mailing_list']
    .agg(lambda s: ", ".join(s.astype(str)))
)

files_agg=files_agg.merge(subs_lists, how='left', on="subsystem")
files_agg["mailing_list"]=files_agg["mailing_list"].fillna("None")


In [129]:
#join subsystems with less then 15 files to speedup treemap processing and visualization (further analysis required to decide n of files)
subs_size=files_agg.groupby("subsystem")["file_count"].transform("sum")
files_agg["sub_group"] = files_agg["subsystem"].astype(str)
files_agg.loc[subs_size < 15, "sub_group"] = "OTHER SUBSYSTEMS"

In [130]:
fig = px.treemap(
    files_agg,
    path=[px.Constant("Linux kernel"), "subsystem", "top_dir", "directory"],
    color="subsystem",
    color_discrete_map={"(?)": "lightgrey"}, #set the background to grey, helping visualization
    maxdepth=2,
    title="Linux Kernel Subsystems treemap",
    custom_data=["mailing_list"]
)

fig.update_traces(marker=dict(cornerradius=2.5))
fig.update_layout(margin=dict(t=50, l=10, r=10, b=10), showlegend=False)

fig.write_html("treemap_kernel_full.html", include_plotlyjs="cdn")

In [131]:
fig.show(renderer="browser")

In [132]:
is_other=files_agg["sub_group"] == "OTHER SUBSYSTEMS"

files_agg["L1"] = files_agg["subsystem"].astype(str).where(~is_other, "OTHER SUBSYSTEMS")
files_agg["L2"] = files_agg["top_dir"].astype(str).where(~is_other, files_agg["subsystem"].astype(str))
files_agg["L3"] = files_agg["directory"].astype(str).where(~is_other, files_agg["top_dir"].astype(str))
files_agg["L4"] = files_agg["directory"].astype(str).where(is_other, None)

In [133]:
fig_agg = px.treemap(
    files_agg,
    path=[px.Constant("Linux kernel"), "L1", "L2", "L3", "L4"],
    color="subsystem",
    values="file_count",
    color_discrete_map={
        "(?)": "lightgrey",  #set the background to grey, helping visualization
        "OTHER SUBSYSTEMS" : "lightgrey"
    },
    maxdepth=2,
    title="Linux Kernel Subsystems treemap",
    custom_data=["mailing_list"]
)

fig_agg.update_traces(marker=dict(cornerradius=2.5))
fig_agg.update_layout(margin=dict(t=50, l=10, r=10, b=10), showlegend=False)

fig_agg.write_html("treemap_kernel_agg.html", include_plotlyjs="cdn")

In [134]:
fig_agg.show(renderer="browser")

### Features addition

Add each file number of LOC (lines of code) and change the dataset to match requirements to use the replication package for the paper "Revisiting Code Authorship in Large FLOSS Projects", avaliable at:  https://doi.org/10.6084/m9.figshare.31549921

In [135]:
#add lines of code of each file
from pathlib import Path

kernel_path=Path("../linux")

def loc_count(file_path):
    try:
        with open(kernel_path/file_path, "rb") as file:
            return sum(1 for l in file)
    except OSError:
        return 0
unique_f=files[["path"]].drop_duplicates().copy()
unique_f["LOC"]=files["path"].map(loc_count)

files = files.drop(columns=["LOC"], errors="ignore").merge(unique_f, on="path", how="left")


In [143]:
display(files)
print(
    "Number of files just in THE REST: ",
    files[files["subsystem"]=="THE REST"].shape[0]
)

,directory,file,subsystem,mailing_list,path,n_subsystem,top_dir,LOC
0,.,.clippy.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.clippy.toml,1,.,21
1,.,.get_maintainer.ignore,THE REST,linux-kernel@vger.kernel.org,.get_maintainer.ignore,1,.,12
2,.,AGENTS.md,THE REST,linux-kernel@vger.kernel.org,AGENTS.md,1,.,168
3,.,.rustfmt.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.rustfmt.toml,1,.,12
4,.,.gitattributes,THE REST,linux-kernel@vger.kernel.org,.gitattributes,1,.,5
...,...,...,...,...,...,...,...,...
184275,virt/kvm,vfio.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/vfio.c,1,virt,364
184276,virt/kvm,kvm_main.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/kvm_main.c,1,virt,6709
184277,virt/lib,Kconfig,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/Kconfig,1,virt,3
184278,virt/lib,irqbypass.c,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/irqbypass.c,1,virt,214


Number of files just in THE REST:  2797


In [137]:
treemap_csv=files.copy()
treemap_csv["identifier"] = "Linux kernel/"+treemap_csv["subsystem"]+"/"+treemap_csv["path"]
treemap_csv["node_name"] = treemap_csv["file"]
treemap_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 184280 entries, 0 to 184279
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   directory     184280 non-null  str   
 1   file          184280 non-null  str   
 2   subsystem     184280 non-null  str   
 3   mailing_list  184280 non-null  str   
 4   path          184280 non-null  str   
 5   n_subsystem   184280 non-null  int64 
 6   top_dir       184280 non-null  object
 7   LOC           184280 non-null  int64 
 8   identifier    184280 non-null  str   
 9   node_name     184280 non-null  str   
dtypes: int64(2), object(1), str(7)
memory usage: 14.1+ MB


In [138]:
treemap_csv

,directory,file,subsystem,mailing_list,path,n_subsystem,top_dir,LOC,identifier,node_name
0,.,.clippy.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.clippy.toml,1,.,21,Linux kernel/RUST/.clippy.toml,.clippy.toml
1,.,.get_maintainer.ignore,THE REST,linux-kernel@vger.kernel.org,.get_maintainer.ignore,1,.,12,Linux kernel/THE REST/.get_maintainer.ignore,.get_maintainer.ignore
2,.,AGENTS.md,THE REST,linux-kernel@vger.kernel.org,AGENTS.md,1,.,168,Linux kernel/THE REST/AGENTS.md,AGENTS.md
3,.,.rustfmt.toml,RUST,"rust-for-linux@vger.kernel.org, linux-kernel@vger.kernel.org",.rustfmt.toml,1,.,12,Linux kernel/RUST/.rustfmt.toml,.rustfmt.toml
4,.,.gitattributes,THE REST,linux-kernel@vger.kernel.org,.gitattributes,1,.,5,Linux kernel/THE REST/.gitattributes,.gitattributes
...,...,...,...,...,...,...,...,...,...,...
184275,virt/kvm,vfio.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/vfio.c,1,virt,364,Linux kernel/KERNEL VIRTUAL MACHINE (KVM)/virt/kvm/vfio.c,vfio.c
184276,virt/kvm,kvm_main.c,KERNEL VIRTUAL MACHINE (KVM),"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/kvm/kvm_main.c,1,virt,6709,Linux kernel/KERNEL VIRTUAL MACHINE (KVM)/virt/kvm/kvm_main.c,kvm_main.c
184277,virt/lib,Kconfig,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/Kconfig,1,virt,3,Linux kernel/VIRT LIB/virt/lib/Kconfig,Kconfig
184278,virt/lib,irqbypass.c,VIRT LIB,"kvm@vger.kernel.org, linux-kernel@vger.kernel.org",virt/lib/irqbypass.c,1,virt,214,Linux kernel/VIRT LIB/virt/lib/irqbypass.c,irqbypass.c


In [139]:
#create tree levels

#root (full kernel)
root_df = pd.DataFrame([{
    "identifier": "Linux kernel",
    "node_name": "Linux kernel",
    "subsystem": "None",
    "n_files": 0,
    "LOC": 0,
}])

#subsystems
subs_df=(
    files[["subsystem"]].drop_duplicates()
    .assign(
        identifier=lambda df: "Linux kernel/"+df["subsystem"],
        node_name=lambda df: df["subsystem"],
        n_files=0,
        LOC=0,
    )
    .merge(subs_lists, on="subsystem", how="left")
)

#top directory
top_dir_df = (
    files[files["top_dir"] != "."][["subsystem", "top_dir"]].drop_duplicates()
    .assign(
        identifier=lambda df: "Linux kernel/" + df["subsystem"] + "/" + df["top_dir"],
        node_name=lambda df: df["top_dir"],
        n_files=0,
        LOC=0,
    )
    .drop(columns=["top_dir"])
)

#subdirectories (subsystems' leaves)
dir_agg = (
    files.groupby(["subsystem", "top_dir", "directory"], as_index=False)
    .agg(n_files=("file", "count"), LOC=("LOC", "sum"))
)

leaves_df = pd.DataFrame({
    "identifier": "Linux kernel/" + dir_agg["subsystem"] + "/" + dir_agg["directory"],
    "node_name": dir_agg["directory"].str.split("/", n=1).str[-1],
    "subsystem": dir_agg["subsystem"],
    "n_files": dir_agg["n_files"],
    "LOC": dir_agg["LOC"],
})

#avoid duplicated identifiers
in_top_dir = (dir_agg["top_dir"] != ".") & (dir_agg["directory"] == dir_agg["top_dir"])
leaves_df.loc[in_top_dir, "identifier"] += "/."
leaves_df.loc[in_top_dir, "node_name"] = "."


#merge all levels
treemap_csv=pd.concat(
    [root_df, subs_df, top_dir_df, leaves_df],
    ignore_index=True
)

In [140]:
treemap_csv

,identifier,node_name,subsystem,n_files,LOC,mailing_list
0,Linux kernel,Linux kernel,None,0,0,NaN
1,Linux kernel/RUST,RUST,RUST,0,0,rust-for-linux@vger.kernel.org
2,Linux kernel/THE REST,THE REST,THE REST,0,0,linux-kernel@vger.kernel.org
3,Linux kernel/CLANG-FORMAT FILE,CLANG-FORMAT FILE,CLANG-FORMAT FILE,0,0,NaN
4,Linux kernel/LICENSES and SPDX stuff,LICENSES and SPDX stuff,LICENSES and SPDX stuff,0,0,linux-spdx@vger.kernel.org
...,...,...,...,...,...,...
29051,Linux kernel/ZSWAP COMPRESSED SWAP CACHING/mm/.,.,ZSWAP COMPRESSED SWAP CACHING,1,1903,NaN
29052,Linux kernel/ZSWAP COMPRESSED SWAP CACHING/tools/testing/selftests/cgroup,testing/selftests/cgroup,ZSWAP COMPRESSED SWAP CACHING,1,892,NaN
29053,Linux kernel/ZTE DINGHAI ETHERNET DRIVER/drivers/net/ethernet/zte,net/ethernet/zte,ZTE DINGHAI ETHERNET DRIVER,2,26,NaN
29054,Linux kernel/ZTE DINGHAI ETHERNET DRIVER/drivers/net/ethernet/zte/dinghai,net/ethernet/zte/dinghai,ZTE DINGHAI ETHERNET DRIVER,5,621,NaN


In [141]:
treemap_csv.to_csv("../data/linux_subsystems.csv", index=False)